### Using a Specific Model (Twitter-RoBERTa)

In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


### Importing libraries

In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm import tqdm # Progress bar library

### Setting up Model and Tokenizer on GPU

In [3]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Move the model to the GPU
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

### Loading Data

In [5]:
# 2. Load Data (Using a sample for demonstration)
# In a real scenario, load your CSV here: df = pd.read_csv(...)
df = pd.read_csv("Tweets.csv")
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [6]:
# 3. The Batch Prediction Function
def get_sentiment_batch(texts, batch_size=32):
    # Lists to store results
    predictions = []
    labels_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

    # Loop through the data in chunks (batches)
    for i in tqdm(range(0, len(texts), batch_size), desc="Processing Batches"):
        batch_texts = texts[i : i + batch_size]

        # Tokenize the batch
        # padding=True ensures all sentences in the batch are the same length
        # truncation=True cuts off extra long tweets
        encoded_input = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True, max_length=128)

        # Move inputs to the GPU
        encoded_input = {key: val.to(device) for key, val in encoded_input.items()}

        # Inference (No Gradient calculation needed for prediction, saves memory)
        with torch.no_grad():
            output = model(**encoded_input)

        # Get scores and move back to CPU to process with Numpy
        scores = output.logits.detach().cpu().numpy()
        scores = softmax(scores, axis=1)

        # Get the label with the highest score for each tweet in the batch
        batch_preds = [labels_map[np.argmax(score)] for score in scores]
        predictions.extend(batch_preds)

    return predictions

# 4. Run it!
print("Starting batch processing...")
# We convert the column to a list for easier batching
all_texts = df['text'].tolist()
df['bert_sentiment'] = get_sentiment_batch(all_texts, batch_size=64)

# 5. Check results
print(df.head())

Starting batch processing...


Processing Batches: 100%|██████████| 229/229 [00:37<00:00,  6.14it/s]

             tweet_id airline_sentiment  airline_sentiment_confidence  \
0  570306133677760513           neutral                        1.0000   
1  570301130888122368          positive                        0.3486   
2  570301083672813571           neutral                        0.6837   
3  570301031407624196          negative                        1.0000   
4  570300817074462722          negative                        1.0000   

  negativereason  negativereason_confidence         airline  \
0            NaN                        NaN  Virgin America   
1            NaN                     0.0000  Virgin America   
2            NaN                        NaN  Virgin America   
3     Bad Flight                     0.7033  Virgin America   
4     Can't Tell                     1.0000  Virgin America   

  airline_sentiment_gold        name negativereason_gold  retweet_count  \
0                    NaN     cairdin                 NaN              0   
1                    NaN    jnar